Please set up your credentials JSON as GCP_CREDENTIALS secrets

In [1]:
import os

# Configuración local (en lugar de google.colab)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gcs.json"
os.environ["BUCKET_URL"] = "gs://ayokestrazoomcamp"

In [2]:
# Install required packages
%pip install -q google-cloud-bigquery google-cloud-storage pyarrow

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import required libraries
from google.cloud import bigquery
from google.cloud import storage
import pandas as pd

## BigQuery Setup
Los archivos Parquet ya están en GCS. Ahora crearemos:
1. Una tabla externa en BigQuery
2. Una tabla regular en BigQuery

In [5]:
from google.cloud import bigquery

# Initialize BigQuery client
client = bigquery.Client()

# Define project and dataset
project_id = client.project
dataset_id = "yellow_taxi_data"
bucket_name = "ayokestrazoomcamp"

# Create dataset if it doesn't exist
dataset_ref = f"{project_id}.{dataset_id}"
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "US"

try:
    client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {dataset_id} created or already exists")
except Exception as e:
    print(f"Error creating dataset: {e}")

Dataset yellow_taxi_data created or already exists


### Step 1: Create External Table from GCS Parquet Files

In [7]:
# Create external table pointing to GCS parquet files
external_table_id = f"{project_id}.{dataset_id}.yellow_taxi_external"

# Configure external data source
external_config = bigquery.ExternalConfig("PARQUET")
external_config.source_uris = [
    f"gs://{bucket_name}/yellow/yellow_tripdata_2024-01.parquet",
    f"gs://{bucket_name}/yellow/yellow_tripdata_2024-02.parquet",
    f"gs://{bucket_name}/yellow/yellow_tripdata_2024-03.parquet",
    f"gs://{bucket_name}/yellow/yellow_tripdata_2024-04.parquet",
    f"gs://{bucket_name}/yellow/yellow_tripdata_2024-05.parquet",
    f"gs://{bucket_name}/yellow/yellow_tripdata_2024-06.parquet",
]

# Create external table
table = bigquery.Table(external_table_id)
table.external_data_configuration = external_config

# Create or replace the table
table = client.create_table(table, exists_ok=True)
print(f"External table {external_table_id} created successfully")

# Query to verify the external table
query = f"""
SELECT COUNT(*) as total_rows
FROM `{external_table_id}`
"""
result = client.query(query).to_dataframe()
print(f"\nTotal rows in external table: {result['total_rows'][0]:,}")

External table kestra-sandbox-2026.yellow_taxi_data.yellow_taxi_external created successfully


/usr/local/python/3.12.1/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Total rows in external table: 20,332,093


In [ ]:
### Step 2: Create Regular (Materialized) Table from External Table

In [8]:
# Create a regular table from the external table
regular_table_id = f"{project_id}.{dataset_id}.yellow_taxi_trips"

# SQL to create regular table from external table
create_table_query = f"""
CREATE OR REPLACE TABLE `{regular_table_id}` AS
SELECT * FROM `{external_table_id}`
"""

# Execute the query
job = client.query(create_table_query)
job.result()  # Wait for the job to complete

print(f"Regular table {regular_table_id} created successfully")

# Verify the regular table
query = f"""
SELECT COUNT(*) as total_rows
FROM `{regular_table_id}`
"""
result = client.query(query).to_dataframe()
print(f"\nTotal rows in regular table: {result['total_rows'][0]:,}")

# Get table info
table_info = client.get_table(regular_table_id)
print(f"Table size: {table_info.num_bytes / (1024**3):.2f} GB")
print(f"Number of rows: {table_info.num_rows:,}")

Regular table kestra-sandbox-2026.yellow_taxi_data.yellow_taxi_trips created successfully

Total rows in regular table: 20,332,093
Table size: 2.72 GB
Number of rows: 20,332,093


### Verify Tables and Sample Query

In [9]:
# Sample query to show data from the tables
query = f"""
SELECT 
    DATE(tpep_pickup_datetime) as pickup_date,
    COUNT(*) as trip_count,
    AVG(trip_distance) as avg_distance,
    AVG(total_amount) as avg_amount
FROM `{regular_table_id}`
GROUP BY pickup_date
ORDER BY pickup_date
LIMIT 10
"""

result = client.query(query).to_dataframe()
print("Sample data from regular table:")
print(result)

Sample data from regular table:
  pickup_date  trip_count  avg_distance  avg_amount
0  2002-12-31          10      4.928000   32.572000
1  2008-12-31           5     12.334000   69.924000
2  2009-01-01           9      7.248889   43.978889
3  2023-12-31          10      2.601000   22.462000
4  2024-01-01       81013      4.396814   30.153719
5  2024-01-02       75519      4.119031   30.220157
6  2024-01-03       82427      3.878400   28.602139
7  2024-01-04      102901      3.310969   27.215586
8  2024-01-05      103178      3.753546   26.446263
9  2024-01-06       97117      3.125827   25.085951


In [10]:
# List all tables in the dataset
tables = client.list_tables(dataset_id)
print(f"\nTables in dataset '{dataset_id}':")
for table in tables:
    print(f"  - {table.table_id} ({table.table_type})")


Tables in dataset 'yellow_taxi_data':
  - yellow_taxi_external (EXTERNAL)
  - yellow_taxi_trips (TABLE)


In [ ]:
### Question: Count Distinct PULocationIDs
Query to count distinct pickup locations and compare data read between external and regular tables

In [11]:
# Query for External Table - count distinct PULocationIDs
query_external = f"""
SELECT COUNT(DISTINCT PULocationID) as distinct_locations
FROM `{external_table_id}`
"""

print("=" * 60)
print("EXTERNAL TABLE QUERY")
print("=" * 60)

# Execute query and get job statistics
job_external = client.query(query_external)
result_external = job_external.result()

# Get the result
df_external = job_external.to_dataframe()
print(f"\nDistinct PULocationIDs: {df_external['distinct_locations'][0]}")

# Get bytes processed (estimate for external table)
bytes_processed_external = job_external.total_bytes_processed
print(f"Data read: {bytes_processed_external / (1024**3):.2f} GB ({bytes_processed_external:,} bytes)")
print(f"Bytes billed: {job_external.total_bytes_billed / (1024**3):.2f} GB")

EXTERNAL TABLE QUERY

Distinct PULocationIDs: 262
Data read: 0.15 GB (162,656,744 bytes)
Bytes billed: 0.15 GB


/usr/local/python/3.12.1/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [12]:
# Query for Regular Table - count distinct PULocationIDs
query_regular = f"""
SELECT COUNT(DISTINCT PULocationID) as distinct_locations
FROM `{regular_table_id}`
"""

print("=" * 60)
print("REGULAR TABLE QUERY")
print("=" * 60)

# Execute query and get job statistics
job_regular = client.query(query_regular)
result_regular = job_regular.result()

# Get the result
df_regular = job_regular.to_dataframe()
print(f"\nDistinct PULocationIDs: {df_regular['distinct_locations'][0]}")

# Get bytes processed
bytes_processed_regular = job_regular.total_bytes_processed
print(f"Data read: {bytes_processed_regular / (1024**3):.2f} GB ({bytes_processed_regular:,} bytes)")
print(f"Bytes billed: {job_regular.total_bytes_billed / (1024**3):.2f} GB")

REGULAR TABLE QUERY

Distinct PULocationIDs: 262
Data read: 0.15 GB (162,656,744 bytes)
Bytes billed: 0.15 GB


In [13]:
# Summary comparison
print("\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"\n{'Metric':<30} {'External Table':<20} {'Regular Table':<20}")
print("-" * 70)
print(f"{'Distinct PULocationIDs':<30} {df_external['distinct_locations'][0]:<20} {df_regular['distinct_locations'][0]:<20}")
print(f"{'Data Read (GB)':<30} {bytes_processed_external / (1024**3):<20.2f} {bytes_processed_regular / (1024**3):<20.2f}")
print(f"{'Data Read (MB)':<30} {bytes_processed_external / (1024**2):<20.2f} {bytes_processed_regular / (1024**2):<20.2f}")
print(f"{'Bytes Billed (GB)':<30} {job_external.total_bytes_billed / (1024**3):<20.2f} {job_regular.total_bytes_billed / (1024**3):<20.2f}")
print("-" * 70)

# Calculate difference
diff_gb = (bytes_processed_external - bytes_processed_regular) / (1024**3)
print(f"\nDifference: External table reads {abs(diff_gb):.2f} GB {'more' if diff_gb > 0 else 'less'} than regular table")


COMPARISON SUMMARY

Metric                         External Table       Regular Table       
----------------------------------------------------------------------
Distinct PULocationIDs         262                  262                 
Data Read (GB)                 0.15                 0.15                
Data Read (MB)                 155.12               155.12              
Bytes Billed (GB)              0.15                 0.15                
----------------------------------------------------------------------

Difference: External table reads 0.00 GB less than regular table
